# OptiCrop: Smart Agricultural Production Optimization Engine

End-to-end notebook: Data Collection -> EDA (Univariate/Bivariate/Multivariate) -> Preprocessing -> Model Building -> Model Saving.

## Epic 2 - Data Collection and Analysis
### Importing the Libraries

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('max_colwidth', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('fivethirtyeight')
plt.rcParams['figure.figsize'] = (12, 8)

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import pickle

### Read the Dataset
Dataset: Crop_recommendation.csv style data (N, P, K, temperature, humidity, ph, rainfall, label).

In [ ]:
df = pd.read_csv('../data/crop_data.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Univariate Analysis
Distribution of individual features.

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()
for i, col in enumerate(features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='seagreen')
    axes[i].set_title(f'Distribution of {col}')
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
sns.countplot(y='label', data=df, order=df['label'].value_counts().index, color='steelblue')
plt.title('Count of Records per Crop')
plt.show()

### Bivariate Analysis
Relationship between humidity and crop label.

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='humidity', y='label', data=df, hue='label', legend=False)
plt.title('Humidity vs Crop Label')
plt.xlabel('Humidity (%)')
plt.show()

### Multivariate Analysis
Relationships across multiple environmental parameters simultaneously.

In [ ]:
sns.pairplot(df.sample(300, random_state=42), hue='label', vars=['N','P','K','temperature','humidity','rainfall'], corner=True, plot_kws={'alpha':0.5, 's':15})
plt.show()

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df[features].corr(), annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Correlation Heatmap of Environmental Features')
plt.show()

## Epic 3 - Data Pre-Processing
### Checking for Null Values

In [ ]:
df.shape
df.info()
df.isnull().sum()

### Handling Outliers
Outliers are visualized using boxplots. Bounds via IQR:

- Upper Bound = Q3 + 1.5 x IQR
- Lower Bound = Q1 - 1.5 x IQR

The boxplot below shows Potassium (K) contains outliers; a log transform is applied to normalize its distribution.

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df[features])
plt.title('Boxplot of Environmental Features (Outlier Check)')
plt.show()

In [ ]:
Q1 = df['K'].quantile(0.25)
Q3 = df['K'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR
print('Lower bound:', lower_bound, ' Upper bound:', upper_bound)

outliers = df[(df['K'] < lower_bound) | (df['K'] > upper_bound)]
print('Number of outliers in K:', outliers.shape[0])

# Log transform to normalize the distribution
df['K_log'] = np.log1p(df['K'])
sns.histplot(df['K_log'], kde=True, color='darkorange')
plt.title('Log-Transformed Potassium (K) Distribution')
plt.show()

### Extracting Seasonal Crops
Crops are grouped by temperature/humidity/rainfall ranges into Summer, Winter, and Rainy season buckets to support season-based recommendations.

In [ ]:
def get_season(row):
    if row['rainfall'] > 150 and row['humidity'] > 80:
        return 'Rainy'
    elif row['temperature'] < 22:
        return 'Winter'
    else:
        return 'Summer'

df['season'] = df.apply(get_season, axis=1)
df.groupby('season')['label'].unique().apply(lambda x: list(x)[:6])

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x='season', data=df, order=['Summer','Winter','Rainy'], palette='Set2')
plt.title('Record Count by Season')
plt.show()

### Splitting Data into Train and Test Sets

In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

X = df[features]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Epic 4 - Model Building
### K-Means Clustering
Used to explore natural groupings of agricultural conditions (unsupervised, exploratory - not used for final prediction).

In [ ]:
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,11), wcss, marker='o', color='crimson')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_train_scaled)
df_clusters = pd.DataFrame({'cluster': cluster_labels, 'crop': y_train.values})
df_clusters.groupby('cluster')['crop'].apply(lambda x: x.value_counts().head(3))

### Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))

### K-Nearest Neighbors, Decision Tree, Random Forest
Additional algorithms trained for comparison, per project requirements.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

dt = DecisionTreeClassifier(random_state=42, max_depth=10)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print('KNN Accuracy:', accuracy_score(y_test, y_pred_knn))
print('Decision Tree Accuracy:', accuracy_score(y_test, y_pred_dt))
print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))

### Evaluating Model Performance and Saving the Best Model
Comparing Accuracy, Precision, Recall, F1-Score for all models.

In [ ]:
results = {}
for name, preds in [('Logistic Regression', y_pred_lr), ('KNN', y_pred_knn),
                     ('Decision Tree', y_pred_dt), ('Random Forest', y_pred_rf)]:
    results[name] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, preds, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_test, preds, average='weighted', zero_division=0),
    }

results_df = pd.DataFrame(results).T.sort_values('Accuracy', ascending=False)
results_df

In [ ]:
best_model_name = results_df.index[0]
print('Best performing model:', best_model_name)

model_map = {'Logistic Regression': log_reg, 'KNN': knn, 'Decision Tree': dt, 'Random Forest': rf}
best_model = model_map[best_model_name]

print(classification_report(y_test, best_model.predict(X_test_scaled)))

In [ ]:
plt.figure(figsize=(12,10))
cm = confusion_matrix(y_test, best_model.predict(X_test_scaled))
sns.heatmap(cm, cmap='Blues', xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Save the Best Model (Pickle)
Saved for reuse in the Flask application without retraining.

In [ ]:
import os
os.makedirs('../model', exist_ok=True)

with open('../model/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('../model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model and scaler saved to ../model/')

### Predict the Best Crop Based on Given Parameters

In [ ]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall):
    with open('../model/model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('../model/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    input_df = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=features)
    scaled = scaler.transform(input_df)
    return model.predict(scaled)[0]

# Example
predict_crop(90, 42, 43, 20.8, 82.0, 6.5, 202.9)